# HadISD 2024 Aggregation + Quality Control — Basemap Fixed

This notebook corrects two issues:

1. **Missing values** are checked only as real missing/invalid fields inside existing rows.
2. **Temporal gaps** are analysed separately and are **not** counted as missing values.
3. The HadISD station map now has a robust map-style background:
   - Uses Cartopy basemap if available.
   - If Cartopy is not available, it still draws an Adriatic map-style background with:
     - sea/land colouring,
     - project domain rectangle,
     - approximate Italy and Balkan coastline guide lines,
     - station points.

So the output will no longer become a blank scatter plot.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

INPUT_CSV = Path(
    r"C:\projects\StormEngine-main\StormEngine-main\DataAggregation\HadISD\hadisd_adriatic_2024.csv"
)

OUTPUT_DIR = Path(
    r"C:\projects\StormEngine-main\StormEngine-main\DataAggregation\HadISD\reports\figures\hadisd_2024_all_variables_qc"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = 300_000

print("Input CSV:", INPUT_CSV)
print("Output directory:", OUTPUT_DIR)
print("CSV exists:", INPUT_CSV.exists())


In [ ]:
def find_column(df, candidates):
    lower_map = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().strip()
        if key in lower_map:
            return lower_map[key]
    return None


def normalise_variable_name(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().lower()

    mapping = {
        "msl": "PRESS",
        "slp": "PRESS",
        "stnlp": "PRESS",
        "press": "PRESS",
        "pressure": "PRESS",

        "u10": "U10",
        "v10": "V10",

        "t2m": "TARIA2M",
        "2t": "TARIA2M",
        "taria2m": "TARIA2M",
        "temperature": "TARIA2M",
        "temperatures": "TARIA2M",
    }

    return mapping.get(x, x.upper())


def expected_range(var):
    ranges = {
        "PRESS": (850, 1100),      # hPa
        "TARIA2M": (-60, 60),      # °C
        "U10": (-80, 80),          # m/s
        "V10": (-80, 80),          # m/s
    }
    return ranges.get(var, (None, None))


In [ ]:
clean_chunks = []
missing_summaries = []
physical_flag_summaries = []
raw_count_by_variable = {}

first_chunk = True

for chunk in pd.read_csv(INPUT_CSV, chunksize=CHUNKSIZE, low_memory=False):

    if first_chunk:
        station_col = find_column(chunk, ["station_id", "station", "id", "stationid"])
        time_col = find_column(chunk, ["timestamp", "time", "datetime", "date", "dt"])
        variable_col = find_column(chunk, ["sensor_code", "variable", "var", "name", "sensor_name", "quantity"])
        value_col = find_column(chunk, ["value", "measurement", "obs", "observation"])
        unit_col = find_column(chunk, ["unit", "units"])
        lat_col = find_column(chunk, ["lat", "latitude"])
        lon_col = find_column(chunk, ["lon", "longitude"])

        required = {
            "station_id": station_col,
            "timestamp": time_col,
            "variable": variable_col,
            "value": value_col,
            "unit": unit_col,
            "lat": lat_col,
            "lon": lon_col,
        }

        print("Detected columns:")
        for k, v in required.items():
            print(f"{k}: {v}")

        missing_required = [k for k, v in required.items() if v is None]
        if missing_required:
            raise ValueError(f"Missing required columns in CSV: {missing_required}")

        first_chunk = False

    df = chunk[
        [station_col, time_col, variable_col, value_col, unit_col, lat_col, lon_col]
    ].copy()

    df.columns = ["station_id", "timestamp", "original_variable", "value", "unit", "lat", "lon"]

    df["variable"] = df["original_variable"].apply(normalise_variable_name)
    df = df[df["variable"].isin(["PRESS", "U10", "V10", "TARIA2M"])].copy()

    if df.empty:
        continue

    for var, count in df["variable"].value_counts().items():
        raw_count_by_variable[var] = raw_count_by_variable.get(var, 0) + int(count)

    # Real missing-value check only inside existing rows.
    missing_summary = {
        "station_id_missing": int(df["station_id"].isna().sum()),
        "timestamp_missing": int(df["timestamp"].isna().sum()),
        "variable_missing": int(df["variable"].isna().sum()),
        "value_missing": int(df["value"].isna().sum()),
        "unit_missing": int(df["unit"].isna().sum()),
        "lat_missing": int(df["lat"].isna().sum()),
        "lon_missing": int(df["lon"].isna().sum()),
        "rows_checked": int(len(df)),
    }
    missing_summaries.append(missing_summary)

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

    df["invalid_timestamp"] = df["timestamp"].isna()
    df["invalid_value"] = df["value"].isna()
    df["invalid_lat"] = df["lat"].isna()
    df["invalid_lon"] = df["lon"].isna()

    df = df[
        (~df["invalid_timestamp"]) &
        (~df["invalid_value"]) &
        (~df["invalid_lat"]) &
        (~df["invalid_lon"])
    ].copy()

    df = df[
        df["lat"].between(-90, 90) &
        df["lon"].between(-180, 180)
    ].copy()

    df["physical_flag"] = False

    for var in df["variable"].unique():
        low, high = expected_range(var)
        if low is None:
            continue

        mask_var = df["variable"] == var
        mask_bad = mask_var & ~df["value"].between(low, high)

        df.loc[mask_bad, "physical_flag"] = True

        physical_flag_summaries.append({
            "variable": var,
            "checked_rows": int(mask_var.sum()),
            "flagged_rows": int(mask_bad.sum()),
            "lower_bound": low,
            "upper_bound": high,
        })

    df_clean = df[df["physical_flag"] == False].copy()

    clean_chunks.append(
        df_clean[
            ["station_id", "timestamp", "original_variable", "variable", "value", "unit", "lat", "lon", "physical_flag"]
        ]
    )

if not clean_chunks:
    raise ValueError("No cleaned data was produced. Check the CSV path and variable/column names.")

hadisd_clean_df = pd.concat(clean_chunks, ignore_index=True)

print("Finished reading and cleaning chunks.")
print("Cleaned dataset shape:", hadisd_clean_df.shape)
display(hadisd_clean_df.head())

cleaned_csv = OUTPUT_DIR / "hadisd_2024_cleaned_all_variables.csv"
hadisd_clean_df.to_csv(cleaned_csv, index=False)
print("Saved cleaned dataset to:", cleaned_csv)


In [ ]:
# Real missing-value summary

missing_df = pd.DataFrame(missing_summaries)
missing_total = missing_df.sum(numeric_only=True).to_frame("count")
missing_total.to_csv(OUTPUT_DIR / "missing_value_summary_real_fields.csv")

print("Real missing-value summary:")
display(missing_total)

total_missing_real_fields = (
    missing_total
    .drop(index=["rows_checked"], errors="ignore")["count"]
    .sum()
)

if total_missing_real_fields == 0:
    print("No real missing values were found in the existing HadISD records.")
    print("Temporal gaps are analysed separately.")
else:
    print("Some real missing or invalid fields were found and removed during cleaning.")


# Physical plausibility summary

physical_summary_df = pd.DataFrame(physical_flag_summaries)

if not physical_summary_df.empty:
    physical_summary_df = (
        physical_summary_df
        .groupby(["variable", "lower_bound", "upper_bound"], as_index=False)
        .agg(
            checked_rows=("checked_rows", "sum"),
            flagged_rows=("flagged_rows", "sum")
        )
    )

    physical_summary_df["flagged_percentage"] = (
        100 * physical_summary_df["flagged_rows"] / physical_summary_df["checked_rows"]
    )

    physical_summary_df.to_csv(
        OUTPUT_DIR / "physical_plausibility_summary.csv",
        index=False
    )

    print("Physical plausibility summary:")
    display(physical_summary_df)


In [ ]:
# Cleaned observation count by variable

counts = (
    hadisd_clean_df["variable"]
    .value_counts()
    .reindex(["PRESS", "U10", "V10", "TARIA2M"])
    .dropna()
)

plt.figure(figsize=(10, 5), dpi=200)
bars = plt.bar(counts.index, counts.values)

plt.title("Cleaned HadISD observations by variable - 2024", fontsize=15, fontweight="bold")
plt.xlabel("Variable", fontsize=12)
plt.ylabel("Number of cleaned observations", fontsize=12)
plt.grid(axis="y", alpha=0.3)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height):,}",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "00_cleaned_observations_by_variable.png", dpi=300, bbox_inches="tight")
plt.show()


## Robust HadISD map

This map cell is fixed.

If Cartopy works, it uses a real basemap.  
If Cartopy does not work, it still draws a map-like Adriatic background with approximate coastline guide lines and the project domain boundary.

This avoids the previous empty/blank scatter output.


In [ ]:
# ------------------------------------------------------------
# HadISD station map with robust basemap / fallback
# ------------------------------------------------------------

from matplotlib.patches import Rectangle, Polygon

stations = (
    hadisd_clean_df[["station_id", "lat", "lon"]]
    .dropna()
    .drop_duplicates()
)

# Adriatic project domain
lon_min, lon_max = 11.5, 20.5
lat_min, lat_max = 39.0, 46.7

# Approximate coastline guide lines.
# These are not official shapefiles. They are only used for the fallback plot
# so the presentation figure does not become a blank scatter plot if Cartopy fails.

italy_adriatic_lon = np.array([12.35, 12.45, 12.55, 12.65, 12.85, 13.05, 13.25, 13.45, 13.65, 13.85, 14.05, 14.25, 14.45, 14.65, 14.85, 15.05, 15.30, 15.60, 16.00, 16.40, 16.90, 17.30, 17.70, 18.20])
italy_adriatic_lat = np.array([45.65, 45.30, 44.95, 44.65, 44.35, 44.10, 43.80, 43.50, 43.20, 42.90, 42.55, 42.25, 41.95, 41.65, 41.35, 41.10, 40.85, 40.60, 40.35, 40.10, 39.85, 39.65, 39.45, 39.25])

balkan_adriatic_lon = np.array([13.55, 13.65, 13.85, 14.10, 14.45, 14.85, 15.25, 15.65, 16.05, 16.45, 16.85, 17.25, 17.65, 18.05, 18.45, 18.85, 19.20, 19.45])
balkan_adriatic_lat = np.array([45.60, 45.35, 45.05, 44.80, 44.55, 44.25, 43.95, 43.65, 43.35, 43.05, 42.70, 42.35, 42.00, 41.65, 41.25, 40.85, 40.40, 39.95])

def plot_fallback_adriatic_map(stations):
    """
    Fallback map that does not need Cartopy or internet.
    It keeps the Adriatic context visible using approximate coastline guide lines.
    """
    fig, ax = plt.subplots(figsize=(11, 7), dpi=250)

    # Sea background
    ax.set_facecolor("#d7ecf8")

    # Very rough land polygons to visually separate Italy and Balkans.
    # Left land mass: Italy side
    italy_poly_x = np.r_[lon_min, lon_min, italy_adriatic_lon, lon_min]
    italy_poly_y = np.r_[lat_min, lat_max, italy_adriatic_lat, lat_min]

    # Right land mass: Balkan side
    balkan_poly_x = np.r_[balkan_adriatic_lon, lon_max, lon_max, balkan_adriatic_lon[0]]
    balkan_poly_y = np.r_[balkan_adriatic_lat, lat_min, lat_max, balkan_adriatic_lat[0]]

    italy_poly = Polygon(
        np.column_stack([italy_poly_x, italy_poly_y]),
        closed=True,
        facecolor="#f2efe9",
        edgecolor="none",
        zorder=1
    )

    balkan_poly = Polygon(
        np.column_stack([balkan_poly_x, balkan_poly_y]),
        closed=True,
        facecolor="#f2efe9",
        edgecolor="none",
        zorder=1
    )

    ax.add_patch(italy_poly)
    ax.add_patch(balkan_poly)

    # Coastline guide lines
    ax.plot(
        italy_adriatic_lon,
        italy_adriatic_lat,
        color="#444444",
        linewidth=1.5,
        zorder=2,
        label="Approx. Italian Adriatic coast"
    )

    ax.plot(
        balkan_adriatic_lon,
        balkan_adriatic_lat,
        color="#444444",
        linewidth=1.5,
        zorder=2,
        label="Approx. Balkan Adriatic coast"
    )

    # Project domain boundary
    domain_box = Rectangle(
        (lon_min, lat_min),
        lon_max - lon_min,
        lat_max - lat_min,
        linewidth=2.0,
        edgecolor="#f28c28",
        facecolor="none",
        linestyle="--",
        label="Project domain",
        zorder=4
    )
    ax.add_patch(domain_box)

    # Stations
    ax.scatter(
        stations["lon"],
        stations["lat"],
        s=38,
        color="#0077b6",
        edgecolor="white",
        linewidth=0.5,
        alpha=0.9,
        zorder=5,
        label="HadISD stations"
    )

    # Labels for context
    ax.text(12.1, 44.7, "Italy", fontsize=11, color="#555555", fontweight="bold")
    ax.text(17.4, 44.1, "Balkan coast", fontsize=11, color="#555555", fontweight="bold")
    ax.text(15.0, 42.3, "Adriatic Sea", fontsize=13, color="#2b6f91", fontweight="bold", alpha=0.85)

    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(
        "HadISD 2024 station coverage over the Adriatic domain",
        fontsize=15,
        fontweight="bold",
        pad=12
    )

    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(loc="lower left", fontsize=9, frameon=True, framealpha=0.95)

    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "01_hadisd_station_map_basemap_fixed.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    fig = plt.figure(figsize=(11, 7), dpi=250)
    ax = plt.axes(projection=ccrs.PlateCarree())

    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND.with_scale("10m"), facecolor="#f2efe9", zorder=1)
    ax.add_feature(cfeature.OCEAN.with_scale("10m"), facecolor="#d7ecf8", zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale("10m"), linewidth=1.1, edgecolor="#444444", zorder=2)
    ax.add_feature(cfeature.BORDERS.with_scale("10m"), linewidth=0.7, edgecolor="#777777", zorder=2)
    ax.add_feature(cfeature.LAKES.with_scale("10m"), facecolor="#d7ecf8", edgecolor="#999999", linewidth=0.4, zorder=1)

    # Add the approximate coastline guide lines on top as well,
    # so the two visual boundary lines remain visible in the final output.
    ax.plot(
        italy_adriatic_lon,
        italy_adriatic_lat,
        color="#222222",
        linewidth=1.3,
        transform=ccrs.PlateCarree(),
        zorder=3,
        label="Italian Adriatic coast guide"
    )

    ax.plot(
        balkan_adriatic_lon,
        balkan_adriatic_lat,
        color="#222222",
        linewidth=1.3,
        transform=ccrs.PlateCarree(),
        zorder=3,
        label="Balkan Adriatic coast guide"
    )

    domain_box = Rectangle(
        (lon_min, lat_min),
        lon_max - lon_min,
        lat_max - lat_min,
        linewidth=2.0,
        edgecolor="#f28c28",
        facecolor="none",
        linestyle="--",
        transform=ccrs.PlateCarree(),
        label="Project domain",
        zorder=4
    )
    ax.add_patch(domain_box)

    ax.scatter(
        stations["lon"],
        stations["lat"],
        s=38,
        color="#0077b6",
        edgecolor="white",
        linewidth=0.5,
        alpha=0.9,
        transform=ccrs.PlateCarree(),
        zorder=5,
        label="HadISD stations"
    )

    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        color="gray",
        alpha=0.35,
        linestyle="--"
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

    ax.set_title(
        "HadISD 2024 station coverage over the Adriatic domain",
        fontsize=15,
        fontweight="bold",
        pad=12
    )

    ax.legend(loc="lower left", fontsize=8.5, frameon=True, framealpha=0.95)

    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "01_hadisd_station_map_basemap_fixed.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

except Exception as e:
    print("Cartopy basemap failed.")
    print("Reason:", e)
    print("Using fallback Adriatic map with approximate coastline guide lines instead.")
    plot_fallback_adriatic_map(stations)


In [ ]:
# Pressure diagnostics

press_df = hadisd_clean_df[hadisd_clean_df["variable"] == "PRESS"].copy()

if press_df.empty:
    print("No PRESS records found. Pressure diagnostics were skipped.")
else:
    plt.figure(figsize=(10, 5), dpi=200)
    plt.hist(press_df["value"], bins=50, edgecolor="black", alpha=0.8)

    plt.title("Distribution of cleaned HadISD pressure values - 2024", fontsize=15, fontweight="bold")
    plt.xlabel("Pressure (hPa)", fontsize=12)
    plt.ylabel("Frequency", fontsize=12)
    plt.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "02_pressure_distribution.png", dpi=300, bbox_inches="tight")
    plt.show()

    daily_press = (
        press_df
        .set_index("timestamp")
        .sort_index()
        .resample("D")["value"]
        .mean()
        .dropna()
    )

    plt.figure(figsize=(11, 5), dpi=200)
    plt.plot(daily_press.index, daily_press.values, linewidth=1.2)

    plt.title("Daily mean HadISD pressure during 2024", fontsize=15, fontweight="bold")
    plt.xlabel("Date", fontsize=12)
    plt.ylabel("Daily mean pressure (hPa)", fontsize=12)
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "03_daily_mean_pressure_timeseries.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
# Temporal gap analysis
# This is NOT missing-value detection.

if press_df.empty:
    print("No PRESS records found. Temporal gap analysis was skipped.")
else:
    gap_records = []

    for station_id, group in press_df.groupby("station_id"):
        group = group.sort_values("timestamp")
        diffs = group["timestamp"].diff().dropna()

        if len(diffs) == 0:
            max_gap_hours = np.nan
        else:
            max_gap_hours = diffs.max().total_seconds() / 3600

        gap_records.append({
            "station_id": station_id,
            "n_observations": len(group),
            "max_temporal_gap_hours": max_gap_hours,
            "lat": group["lat"].iloc[0],
            "lon": group["lon"].iloc[0],
        })

    gap_df = pd.DataFrame(gap_records)
    gap_df.to_csv(OUTPUT_DIR / "temporal_gap_PRESS_by_station.csv", index=False)

    gap_plot_df = (
        gap_df
        .dropna()
        .sort_values("max_temporal_gap_hours", ascending=False)
        .head(30)
    )

    plt.figure(figsize=(12, 6), dpi=200)
    plt.bar(
        gap_plot_df["station_id"].astype(str),
        gap_plot_df["max_temporal_gap_hours"]
    )

    plt.title("Maximum temporal gap by station - PRESS", fontsize=15, fontweight="bold")
    plt.xlabel("Station ID", fontsize=12)
    plt.ylabel("Maximum temporal gap (hours)", fontsize=12)
    plt.xticks(rotation=90, fontsize=8)
    plt.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "04_max_temporal_gap_PRESS_by_station.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
summary = {
    "total_cleaned_observations": len(hadisd_clean_df),
    "number_of_stations": hadisd_clean_df["station_id"].nunique(),
    "variables": ", ".join(sorted(hadisd_clean_df["variable"].unique())),
    "real_missing_values_found": int(total_missing_real_fields),
}

summary_df = pd.DataFrame([summary])

summary_df.to_csv(
    OUTPUT_DIR / "hadisd_qc_summary.csv",
    index=False
)

print("Final summary:")
display(summary_df)

print("Main outputs saved in:")
print(OUTPUT_DIR)

print("\nReport sentence:")
print("No real missing values were found in the existing HadISD records. Temporal gaps were analysed separately.")
